# Role Adherence Metric Example

This notebook demonstrates how to use the **RoleAdherence** metric from Gaussia to evaluate whether an AI assistant consistently adheres to its defined role across a multi-turn conversation.

The metric implements the formula:

$$\text{RoleAdherence}(R, T) = \frac{1}{n} \sum_{i=1}^{n} \text{adhere}(t_i, T_{<i}, R)$$

Where each turn $t_i$ is evaluated using the full conversation history $T_{<i}$ and the role definition $R$ — without requiring ground-truth reference responses.

### Dataset

The example uses a **FinTrack** digital banking support scenario with two sessions:

| Session | Description |
|---|---|
| `session_adherent_001` | All 4 turns fully adhere to the role (scope, tone, constructive behavior) |
| `session_violations_002` | 4 turns mixing adherent responses with a scope violation (investment advice), a tone violation (dismissive language), and a constructive failure (no next steps offered) |

## Installation

Este proyecto usa `uv`. La celda siguiente instala `langchain-groq` en el venv del repo.

Si preferís instalarlo desde la terminal antes de abrir Jupyter:

```bash
uv add langchain-groq
```

In [1]:
!uv add langchain-groq -q

## Setup

Import the required modules and configure your API key.

In [2]:
import sys
from pathlib import Path

# Add examples directory to path for helpers import
sys.path.insert(0, str(Path("../..").resolve()))

from helpers.retriever import LocalRetriever
from langchain_groq import ChatGroq

from gaussia.metrics.role_adherence import LLMJudgeStrategy, RoleAdherence

/Users/frino/Desktop/Alquimia/Gaussia/pygaussia/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import getpass

GROQ_API_KEY = getpass.getpass("Enter your Groq API key: ")

## Initialize the Judge Model

The `LLMJudgeStrategy` uses any LangChain-compatible chat model to evaluate each turn against the role definition. The `binary=True` default prompts the judge to return a hard classification (1 = adheres, 0 = violates) rather than a continuous score.

In [4]:
judge_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.0,
)

scoring_strategy = LLMJudgeStrategy(
    model=judge_model,
    binary=True,   # each turn returns 1 (adherent) or 0 (violation)
    verbose=True,
)

## Run the Role Adherence Metric

The metric reads `chatbot_role` from each session in the dataset and evaluates every turn in context of the full prior conversation history.

Key parameters:
- `binary=True` — per-turn output is `{0, 1}`; session score = proportion of adherent turns
- `strict_mode=False` — session is adherent if `mean >= threshold` (not all-or-nothing)
- `threshold=0.5` — minimum session score to classify as adherent
- `include_reason=True` — include the judge's justification in each turn's output

In [5]:
metrics = RoleAdherence.run(
    LocalRetriever,
    dataset_path="dataset_role_adherence.json",
    scoring_strategy=scoring_strategy,
    binary=True,
    strict_mode=False,
    threshold=0.5,
    include_reason=True,
    verbose=True,
)

2026-04-22 12:49:18,829 - gaussia.utils.logging - INFO - Starting to process dataset
2026-04-22 12:49:18,829 - gaussia.utils.logging - DEBUG - QA ID: s1_t1
2026-04-22 12:49:20,007 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:20,023 - gaussia.utils.logging - DEBUG - Role adherence score: 1.0, adherent: True
2026-04-22 12:49:20,024 - gaussia.utils.logging - DEBUG - QA ID: s1_t2
2026-04-22 12:49:21,402 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:21,404 - gaussia.utils.logging - DEBUG - Role adherence score: 1.0, adherent: True
2026-04-22 12:49:21,405 - gaussia.utils.logging - DEBUG - QA ID: s1_t3
2026-04-22 12:49:22,623 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:22,626 - gaussia.utils.logging - DEBUG - Role adherence score: 1.0, adherent: True
2026-04-22 12:49:22,627 

## Session-Level Results

Each `RoleAdherenceMetric` contains:
- `role_adherence`: proportion of turns the assistant adhered to the role (in binary mode)
- `adherent`: whether the session passes the threshold overall
- `turns`: per-turn breakdown with individual scores and reasons

In [6]:
for metric in metrics:
    status = "✓ ADHERENT" if metric.adherent else "✗ NON-ADHERENT"
    print(f"Session: {metric.session_id}")
    print(f"  Role Adherence Score: {metric.role_adherence:.2f}  [{status}]")
    print(f"  Turns evaluated: {metric.n_turns}")
    print()

Session: session_adherent_001
  Role Adherence Score: 1.00  [✓ ADHERENT]
  Turns evaluated: 4

Session: session_violations_002
  Role Adherence Score: 0.50  [✓ ADHERENT]
  Turns evaluated: 4



## Per-Turn Breakdown

Drill into each turn to see where violations occurred and why the judge flagged them.

In [7]:
for metric in metrics:
    print(f"=== {metric.session_id} ===")
    for turn in metric.turns:
        adherent_label = "✓" if turn.adherent else "✗"
        print(f"  [{adherent_label}] Turn {turn.qa_id}  score={turn.adherence_score:.0f}")
        if turn.reason:
            print(f"      Reason: {turn.reason}")
    print()

=== session_adherent_001 ===
  [✓] Turn s1_t1  score=1
      Reason: The response stays within scope, maintains a professional tone, and offers a constructive solution to the user's issue.
  [✓] Turn s1_t2  score=1
      Reason: The assistant's response stays within scope, maintains a professional tone, and exhibits constructive behavior.
  [✓] Turn s1_t3  score=1
      Reason: The assistant stays within scope, maintains a professional tone, and offers proactive solutions.
  [✓] Turn s1_t4  score=1
      Reason: The assistant's response stays within scope, maintains a professional tone, and exhibits constructive behavior.

=== session_violations_002 ===
  [✓] Turn s2_t1  score=1
      Reason: The response stays within scope, maintains a professional tone, and exhibits constructive behavior.
  [✗] Turn s2_t2  score=0
      Reason: The response violates the scope by providing investment advice, fails to maintain a professional tone, and lacks constructive behavior.
  [✗] Turn s2_t3  scor

## Summary Statistics

In [8]:
total_turns = sum(m.n_turns for m in metrics)
adherent_turns = sum(t.adherent for m in metrics for t in m.turns)
adherent_sessions = sum(1 for m in metrics if m.adherent)
avg_score = sum(m.role_adherence for m in metrics) / len(metrics)

print(f"Sessions evaluated : {len(metrics)}")
print(f"Adherent sessions  : {adherent_sessions}/{len(metrics)}")
print(f"Turns evaluated    : {total_turns}")
print(f"Adherent turns     : {adherent_turns}/{total_turns}")
print(f"Average score      : {avg_score:.2f}")

Sessions evaluated : 2
Adherent sessions  : 2/2
Turns evaluated    : 8
Adherent turns     : 6/8
Average score      : 0.75


## Continuous Mode

Set `binary=False` on the strategy to ask the judge for a graduated score `[0, 1]` instead of a hard classification. Useful when you want to measure *degree* of adherence rather than pass/fail.

In [9]:
continuous_strategy = LLMJudgeStrategy(
    model=judge_model,
    binary=False,
    verbose=False,
)

metrics_continuous = RoleAdherence.run(
    LocalRetriever,
    dataset_path="dataset_role_adherence.json",
    scoring_strategy=continuous_strategy,
    binary=False,
    threshold=0.5,
    include_reason=True,
)

for metric in metrics_continuous:
    print(f"{metric.session_id}: role_adherence={metric.role_adherence:.3f}")
    for turn in metric.turns:
        print(f"  {turn.qa_id}: {turn.adherence_score:.3f}")

2026-04-22 12:49:30,488 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:31,437 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:32,237 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:33,175 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:34,554 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:35,732 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:37,159 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:38,011 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 

session_adherent_001: role_adherence=1.000
  s1_t1: 1.000
  s1_t2: 1.000
  s1_t3: 1.000
  s1_t4: 1.000
session_violations_002: role_adherence=0.550
  s2_t1: 1.000
  s2_t2: 0.000
  s2_t3: 0.200
  s2_t4: 1.000


## Strict Mode

With `strict_mode=True`, a session is only marked `adherent=True` if **every single turn** passes. Useful for high-compliance scenarios where a single violation is unacceptable.

In [10]:
strict_strategy = LLMJudgeStrategy(model=judge_model, binary=True, verbose=False)

metrics_strict = RoleAdherence.run(
    LocalRetriever,
    dataset_path="dataset_role_adherence.json",
    scoring_strategy=strict_strategy,
    binary=True,
    strict_mode=True,
    threshold=0.5,
)

for metric in metrics_strict:
    status = "PASS" if metric.adherent else "FAIL"
    print(f"{metric.session_id}: {status}  (score={metric.role_adherence:.2f})")

2026-04-22 12:49:39,071 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:40,837 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:42,452 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:43,595 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:44,650 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:45,864 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:47,487 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 12:49:48,847 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 

session_adherent_001: PASS  (score=1.00)
session_violations_002: FAIL  (score=0.50)
